# Fındık Görüntüleri Sınıflandırma Projesinde DenseNet-121 Mimarisi Kullanımı

Bu proje, derin öğrenme yöntemleri kullanılarak fındık görüntülerinin otonom olarak sınıflandırılmasını amaçlamaktadır. Proje kapsamında MVTec AD fındık veri seti kullanılmış olup, görüntülerdeki kusurlar (çatlak, kesik, delik, baskı hatası) ve sağlam fındıklar ayırt edilmektedir.

## Proje Kapsamı ve Gereksinimler:
- **Veri Seti:** Hazelnut (crack, cut, good, hole, print).
- **Mühendislik Yaklaşımı:** Transfer Learning (DenseNet-121).
- **Veri Artırımı:** Mevcut görsel verisi 5 katına çıkarılmıştır.
- **Model Parametreleri:** Dropout katmanı entegrasyonu ve 8 epok eğitim (Hızlandırılmış).
- **Metrikler:** Accuracy, Precision, Recall ve F1-Score.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader, Subset, Dataset
import numpy as np
import matplotlib.pyplot as plt
import os
import pandas as pd
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, precision_score, recall_score, f1_score
import seaborn as sns
from sklearn.model_selection import train_test_split
from PIL import Image

# Uyarıları kapatma
import warnings
warnings.filterwarnings('ignore')

## 1. Veri Setinin Hazırlanması

Veri seti `hazel` dizininden okunarak tüm sınıflar tek bir yapıda toplanmış ve ardından %80 eğitim, %20 test olacak şekilde ayrılmıştır. Eğitim verisi, veri artırımı (augmentation) teknikleri kullanılarak 5 katına çıkarılmıştır.

In [ ]:
def collect_data(base_path):
    data = []
    for root_dir in ['train', 'test']:
        dir_path = os.path.join(base_path, root_dir)
        if not os.path.exists(dir_path): continue
        
        for label in os.listdir(dir_path):
            label_path = os.path.join(dir_path, label)
            if os.path.isdir(label_path):
                for img_name in os.listdir(label_path):
                    if img_name.endswith('.png'):
                        data.append((os.path.join(label_path, img_name), label))
    return data

base_path = r'C:/Users/user/Desktop/hazel'
all_data = collect_data(base_path)

df = pd.DataFrame(all_data, columns=['path', 'label'])
class_to_idx = {label: i for i, label in enumerate(sorted(df['label'].unique()))}
df['label_idx'] = df['label'].map(class_to_idx)

print(f"Toplam Görsel Sayısı: {len(df)}")

train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label_idx'])

class HazelnutDataset(Dataset):
    def __init__(self, dataframe, transform=None, multiplier=1):
        self.dataframe = dataframe
        self.transform = transform
        self.multiplier = multiplier
        self.data = dataframe.values

    def __len__(self):
        return len(self.data) * self.multiplier

    def __getitem__(self, idx):
        real_idx = idx % len(self.data)
        img_path, _, label_idx = self.data[real_idx]
        image = Image.open(img_path).convert('RGB')
        if self.transform: image = self.transform(image)
        return image, label_idx

train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomRotation(30),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

test_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

train_dataset = HazelnutDataset(train_df, transform=train_transforms, multiplier=5)
test_dataset = HazelnutDataset(test_df, transform=test_transforms, multiplier=1)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

print(f"Eğitim Kümesi (5x Artırılmış): {len(train_dataset)}")
print(f"Test Kümesi: {len(test_dataset)}")

## 2. DenseNet-121 Model Mimarisinin Yapılandırılması

Transfer Learning yöntemiyle eğitilmiş DenseNet-121 modeli kullanılmıştır. Bu mimari, katmanlar arasındaki yoğun bağlantılar sayesinde özellik iletimini optimize eder. Eğitim hızı için temel katmanlar dondurulmuş ve son katman (classifier) fındık sınıflarına göre güncellenmiştir.

In [ ]:
model = models.densenet121(weights='IMAGENET1K_V1')

# Özellik çıkarıcı katmanları dondurma
for param in model.parameters():
    param.requires_grad = False

# Classifier kısmını değiştirme
num_ftrs = model.classifier.in_features
model.classifier = nn.Sequential(
    nn.Linear(num_ftrs, 512),
    nn.ReLU(),
    nn.Dropout(0.4),
    nn.Linear(512, len(class_to_idx))
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.classifier.parameters(), lr=0.001)

print(f"Cihaz: {device}")

## 3. Model Eğitimi (8 Epok)

Eğitim süreci 8 epok boyunca takip edilmiş, her adımda kayıp ve doğruluk değerleri kaydedilmiştir.

In [ ]:
epochs = 8
history = {'train_loss': [], 'test_loss': [], 'train_acc': [], 'test_acc': []}

for epoch in range(epochs):
    model.train()
    train_loss, train_correct = 0.0, 0
    
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        train_correct += torch.sum(preds == labels.data)
        
    epoch_loss = train_loss / len(train_dataset)
    epoch_acc = train_correct.double() / len(train_dataset)
    
    model.eval()
    test_loss, test_correct = 0.0, 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            test_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            test_correct += torch.sum(preds == labels.data)
            
    val_loss = test_loss / len(test_dataset)
    val_acc = test_correct.double() / len(test_dataset)
    
    history['train_loss'].append(epoch_loss)
    history['test_loss'].append(val_loss)
    history['train_acc'].append(epoch_acc.item())
    history['test_acc'].append(val_acc.item())
    
    print(f'Epoch {epoch+1}/{epochs} | Train Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f} | Test Loss: {val_loss:.4f} Acc: {val_acc:.4f}')

## 4. Performans Analizi ve Görselleştirme

Eğitim sürecindeki başarım ve kayıp değişimleri grafikte gösterilmiştir.

In [ ]:
plt.figure(figsize=(14, 5))
plt.subplot(1, 2, 1)
plt.plot(history['train_acc'], label='Eğitim Doğruluğu')
plt.plot(history['test_acc'], label='Test Doğruluğu')
plt.title('DenseNet-121 Doğruluk Grafiği')
plt.xlabel('Epok')
plt.ylabel('Doğruluk')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history['train_loss'], label='Eğitim Kaybı')
plt.plot(history['test_loss'], label='Test Kaybı')
plt.title('DenseNet-121 Kayıp Grafiği')
plt.xlabel('Epok')
plt.ylabel('Kayıp')
plt.legend()
plt.show()

## 5. Değerlendirme Metrikleri

Test verisi üzerinde elde edilen detaylı performans raporu ve karmaşıklık matrisi aşağıdadır.

In [ ]:
y_true, y_pred = [], []
model.eval()

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, preds = torch.max(outputs, 1)
        y_true.extend(labels.cpu().numpy())
        y_pred.extend(preds.cpu().numpy())

class_names = list(class_to_idx.keys())

print("Sektörel Standart Metrik Raporu (DenseNet-121):")
print(classification_report(y_true, y_pred, target_names=class_names))

metrics = {
    "Metrik": ["Accuracy", "Precision (Weighted)", "Recall (Weighted)", "F1 Score (Weighted)"],
    "Değer": [
        accuracy_score(y_true, y_pred),
        precision_score(y_true, y_pred, average='weighted'),
        recall_score(y_true, y_pred, average='weighted'),
        f1_score(y_true, y_pred, average='weighted')
    ]
} 
metrics_df = pd.DataFrame(metrics)
display(metrics_df)

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=class_names, yticklabels=class_names, cmap='OrRd')
plt.xlabel('Tahmini Sınıf')
plt.ylabel('Gerçek Sınıf')
plt.title('Karmaşıklık Matrisi (DenseNet-121)')
plt.show()

## 6. Sonuç ve Değerlendirme

DenseNet-121 mimarisi kullanılarak gerçekleştirilen fındık sınıflandırma projesinde, yoğun bağlantı yapısının avantajları gözlemlenmiştir. 8 epok gibi kısa bir sürede model yüksek başarı oranlarına ulaşmıştır. Veri artırımı ve Dropout kullanımı sayesinde modelin genelleme başarısı korunmuştur.

ResNet-18 ile karşılaştırıldığında DenseNet-121, parametre verimliliği ve özellik tekrar kullanımı konularında güçlü bir alternatif sunmaktadır.